# IRWA Final Project — Part 2: Indexing and Evaluation

* **Part 1 – Indexing:** inverted index, conjunctive (AND) queries, five test queries and TF-IDF ranking.
* **Part 2 – Evaluation:** metrics, results on the validation labels, and our own relevance judgments.

The code is in `irwa_indexing.py` (adapted from Practice Session 1) and `irwa_evaluation.py` (adapted from Practice
Session 2). Queries are cleaned with `preprocess_text` from Part 1, the same function used for the products.

In [13]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import nltk
from IPython.display import display

nltk.download("stopwords", quiet=True)

sys.path.append("../part_1")       # irwa_preprocessing.py (Part 1)
import irwa_preprocessing as pp
import irwa_indexing as ix
import irwa_evaluation as ev

DATA = Path("../../data")          # dataset folder (see the main README)

## 0. Load the processed corpus

`processed_corpus.pkl` is written by the Part 1 notebook (one row per product, with the processed terms of each field).

In [14]:
corpus = pd.read_pickle(DATA / "processed_corpus.pkl")
labels = pd.read_csv(DATA / "validation_labels.csv")
VALIDATION_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
N = len(corpus)
print(f"{N:,} products")

28,080 products


---
# PART 1 — Indexing

## 1.1 Inverted index (point 1)

We index `all_terms` (title + product_details + sub_category + brand + description), as decided in Part 1.
As in the lab (Figure 2), each term maps to the products that contain it and its positions:
`{term: [[pid, [positions]], ...]}`. `ix.compute_tfidf` then computes tf, df and idf from the index.

In [15]:
start = time.time()
index = ix.create_index(ix.build_docs_terms(corpus, "all_terms"))
tf, df, idf = ix.compute_tfidf(index, N)
print(f"index built in {time.time() - start:.1f} s | {len(index):,} terms")

pid = "TKPFCZ9EA7H5FYZH"
print(corpus.loc[pid, "title"])
for term in ["track", "cotton"]:
    i, posting = next((i, p) for i, p in enumerate(index[term]) if p[0] == pid)
    print(f"  {term!r}: positions {list(posting[1])}, tf = {tf[term][i]}, df = {df[term]:,}, idf = {idf[term]}")

index built in 15.2 s | 6,343 terms
Solid Women Multicolor Track Pants
  'track': positions [3], tf = 0.1414, df = 1,815, idf = 2.739
  'cotton': positions [8, 19], tf = 0.2828, df = 21,723, idf = 0.2567


Queries are **conjunctive (AND)**: `ix.search` returns only the products that contain every query term
(intersection of the posting lists).

## 1.2 Test queries (point 2)

Most frequent terms in the titles, which are written the way a shopper searches:

In [16]:
title_df = pd.Series([t for terms in corpus.title_terms for t in set(terms)]).value_counts()
popular = title_df.head(25).to_frame("products (titles)")
popular["idf"] = [round(idf[t], 2) for t in popular.index]
display(popular.astype(object).T)

,women,men,tshirt,neck,solid,round,print,pack,blue,fit,...,multicolor,collar,white,sleev,full,grey,regular,cotton,sweatshirt,dark
products (titles),13041,13021,12247,11952,9215,8301,8106,4787,4282,3923,...,2370,2360,2168,2030,2011,1913,1754,1692,1343,1333
idf,0.71,0.68,0.77,0.66,0.85,1.08,1.08,1.54,1.29,0.94,...,1.76,2.05,1.96,0.44,1.36,2.22,0.75,0.26,2.98,2.41


Gender, product type, colour and style are the most popular terms. Our first queries used only these terms and
almost every result was relevant, so each final query starts from popular terms and **adds one constraint that
keyword matching handles badly**:

| # | query | what makes it hard |
|---|---|---|
| 3 | *men white formal shirt slim fit* | four constraints; `formal` is often only in the description or the product details, so casual shirts match |
| 4 | *women floral print shirt* | unstitched "Shirt Fabric" uses the same words as a shirt |
| 5 | *women black track pants with zip pockets* | zip pockets are only mentioned in the description |
| 6 | *women pure cotton t-shirt pack of 3* | Part 1 removes numbers, so the "3" is lost |
| 7 | *black running shoes for men* | the sub_category "Men's Footwear" also contains women's shoes |

In [17]:
OUR_QUERIES = {
    3: "men white formal shirt slim fit",
    4: "women floral print shirt",
    5: "women black track pants with zip pockets",
    6: "women pure cotton t-shirt pack of 3",
    7: "black running shoes for men",
}
ALL_QUERIES = {**VALIDATION_QUERIES, **OUR_QUERIES}

display(pd.DataFrame([{"query_id": qid, "query": query, "terms": pp.preprocess_text(query),
                       "AND matches": len(ix.search(pp.preprocess_text(query), index))}
                      for qid, query in ALL_QUERIES.items()]).set_index("query_id"))

,query,terms,AND matches
query_id,,,
1,women full sleeve sweatshirt cotton,"[women, full, sleev, sweatshirt, cotton]",566
2,men slim jeans blue,"[men, slim, jean, blue]",225
3,men white formal shirt slim fit,"[men, white, formal, shirt, slim, fit]",25
4,women floral print shirt,"[women, floral, print, shirt]",35
5,women black track pants with zip pockets,"[women, black, track, pant, zip, pocket]",11
6,women pure cotton t-shirt pack of 3,"[women, pure, cotton, tshirt, pack]",140
7,black running shoes for men,"[black, run, shoe, men]",28


## 1.3 TF-IDF ranking (point 3)

Same model as the lab: the score of a product is the dot product between the TF-IDF vectors of the query and of the
product. `ix.search_tf_idf` returns the AND matches as `(pid, score)`, sorted by score (ties broken by `pid`).

In [18]:
SHOW = ["title", "brand_full", "sub_category", "selling_price_num", "average_rating_num"]

for qid, query in ALL_QUERIES.items():
    ranked = ix.search_tf_idf(pp.preprocess_text(query), index, tf, idf)
    top = pd.DataFrame(ranked[:10], columns=["pid", "score"]).join(corpus[SHOW], on="pid")
    top.index = np.arange(1, len(top) + 1)
    print(f"Query {qid}: '{query}' -> {len(ranked)} matches")
    display(top.round({"score": 3}))

Query 1: 'women full sleeve sweatshirt cotton' -> 566 matches


,pid,score,title,brand_full,sub_category,selling_price_num,average_rating_num
1,SWSFMTNHE2PCYWFZ,2.803,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,854.0,4.2
2,SWSFMTNHEM5SNHVZ,2.659,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,854.0,4.2
3,SWSFMF3AUHYGMGYF,2.452,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,930.0,4.2
4,SWSFMF3AVA9FPFDJ,2.452,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,835.0,4.2
5,SWSFMKGBF7FWZ3Q4,2.420,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,635.0,4.2
6,SWSFMJGS8HVBPEH6,2.385,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,759.0,4.3
7,SWSFMJGTGZMJXK5H,2.385,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,949.0,4.2
8,SWSFMJGTQFMZGZQH,2.385,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,759.0,4.2
9,SWSFMJGTVFW3NAXZ,2.385,Full Sleeve Solid Women Sweatshirt,arbour,Winter Wear,759.0,4.2
10,SWSFZVTNGM4HG8BC,2.374,Full Sleeve Printed Women Sweatshirt,levis,Winter Wear,1042.0,4.2


Query 2: 'men slim jeans blue' -> 225 matches


,pid,score,title,brand_full,sub_category,selling_price_num,average_rating_num
1,JEAF2QF6HNZHDXVX,2.122,Slim Men Dark Blue Jeans,absolute,Bottomwear,856.0,3.8
2,JEAFRAQXEKGUPNUN,2.122,Slim Men Blue Jeans,levis,Bottomwear,3999.0,NaN
3,JEAFJM2P2FQQJF3C,2.089,Slim Men Dark Blue Jeans,absolute,Bottomwear,589.0,4.1
4,JEAFDZRVABZZ2PQ3,1.969,Stretchable Slim Men Blue Jeans (Pack of 2),nebraska,Bottomwear,1275.0,3.9
5,JEAFEC2GEMGBWHA5,1.911,Slim Men Light Blue Jeans,nebraska,Bottomwear,629.0,3.9
6,JEAFDHHGY3HBXEMP,1.897,Stretchable Regular Men Blue Jeans,nebraska,Bottomwear,629.0,3.6
7,JEAFSKYHDYZK5SHZ,1.893,Slim Men Blue Jeans,ecko unltd,Bottomwear,899.0,2.5
8,JEAFSKYHRVZSABPR,1.893,Slim Men Blue Jeans,ecko unltd,Bottomwear,974.0,3.3
9,JEAFSKYHTE76YWH7,1.893,Slim Men Blue Jeans,ecko unltd,Bottomwear,974.0,2.5
10,JEAFDWC8NGRWDHRS,1.884,Slim Men Blue Jeans,nebraska,Bottomwear,649.0,3.8


Query 3: 'men white formal shirt slim fit' -> 25 matches


,pid,score,title,brand_full,sub_category,selling_price_num,average_rating_num
1,SHTFDYRQRYGTW4PX,2.038,Men Slim Fit Printed Button Down Collar Formal Shirt,mods,Topwear,799.0,3.7
2,SHTFW52ZURHFKNTZ,1.998,Men Slim Fit Printed Button Down Collar Formal Shirt,truemoda,Topwear,758.0,4.2
3,SHTF6VCHUQ8HKQPE,1.869,Men Slim Fit Solid Casual Shirt,jai textiles,Topwear,411.0,3.4
4,SHTF98P8MTY7ZXAZ,1.846,Men Slim Fit Solid Spread Collar Formal Shirt,dudlind,Topwear,799.0,4.4
5,SHTFEBYMF9RTZYKG,1.807,Men Slim Fit Solid Spread Collar Casual Shirt,purple state,Topwear,561.0,2.9
6,SHTFEBYMGU47WWZG,1.807,Men Slim Fit Solid Spread Collar Casual Shirt,purple state,Topwear,561.0,2.9
7,SHTFEBYMHVXRHZCZ,1.807,Men Slim Fit Solid Spread Collar Casual Shirt,purple state,Topwear,606.0,1.0
8,TROFYFX6XK4HYZFN,1.769,Slim Fit Men White Cotton Blend Trousers,shreeram,Bottomwear,599.0,1.9
9,SHTFDYTZGHNN2JQ9,1.735,Men Slim Fit Polka Print Button Down Collar Casual Shirt,mods,Topwear,799.0,3.9
10,SHTFDYMU7MCMFCT9,1.726,Men Slim Fit Geometric Print Button Down Collar Casual Shirt,mods,Topwear,799.0,3.6


... (4 outputs left out)

Query 6: 'women pure cotton t-shirt pack of 3' -> 140 matches


,pid,score,title,brand_full,sub_category,selling_price_num,average_rating_num
1,TSHFWAWKKGYZSNHU,0.583,Solid Women Round Neck Grey T-Shirt (Pack of 5),scott international,Topwear,799.0,4.2
2,TSHFWBWECMBZ9D8A,0.582,Solid Women Round Neck Black T-Shirt (Pack of 3),scott international,Topwear,499.0,4.1
3,TSHFWBWERWBHYYJG,0.581,Solid Women Round Neck Grey T-Shirt (Pack of 3),scott international,Topwear,499.0,4.1
4,TSHFWAWKK2GNAQJA,0.577,Solid Women Round Neck Maroon T-Shirt (Pack of 5),scott international,Topwear,799.0,4.2
5,TSHFWBWEGZEDCABK,0.577,Solid Women Round Neck Maroon T-Shirt (Pack of 3),scott international,Topwear,499.0,4.1
6,TSHFWBWEV3J285KG,0.576,Solid Women Round Neck Blue T-Shirt (Pack of 3),scott international,Topwear,499.0,4.1
7,TSHFWAWKHFZDGSVX,0.572,Solid Women Round Neck Multicolor T-Shirt (Pack of 5),scott international,Topwear,799.0,4.2
8,TSHFWAWKMVHVWC5P,0.570,Solid Women Round Neck Multicolor T-Shirt (Pack of 5),scott international,Topwear,799.0,4.2
9,TSHFWBWEXX3N9FHZ,0.570,"Solid Women Round Neck Blue, Maroon T-Shirt (Pack of 3)",scott international,Topwear,499.0,4.1
10,TSHFXMBEFHXZZZZ5,0.467,Typography Women Round Neck Multicolor T-Shirt (Pack of 3),attiitude,Topwear,1199.0,2.3


Query 7: 'black running shoes for men' -> 28 matches


,pid,score,title,brand_full,sub_category,selling_price_num,average_rating_num
1,SHOFVXG5XR59AFTH,5.392,Lace Up For Men (Black),bonexy,Men's Footwear,474.0,3.6
2,SHOFN3EJHXHGSGUC,5.287,RS-7250 Running Shoes For Men (Black),vector,Men's Footwear,990.0,3.9
3,SHOFVHY8MAYUPTZF,5.174,Loafers For Men (Black),bonexy,Men's Footwear,474.0,3.8
4,SHOFVJX2TP8FRR7P,4.992,Lace Up For Men (Black),bonexy,Men's Footwear,474.0,3.8
5,SHOFUH52V5AVNHMN,4.959,Loafers For Men (Black),bonexy,Men's Footwear,474.0,3.9
6,SHOFX2G6FHVNAHZN,4.881,Loafers For Women (Black),bonexy,Men's Footwear,474.0,4.0
7,SHOFZP2FHH8Z6ZHC,4.636,Running Shoes For Women (Black),bonexy,Men's Footwear,474.0,4.0
8,SHOFTZFQP8W9G86F,4.515,Running Shoes For Women (Black),bonexy,Men's Footwear,341.0,4.1
9,SHOEKNRJ34UCMAPA,4.509,"SPRINT Running Shoes For Men (Black, Grey)",vector,Men's Footwear,490.0,3.7
10,SHOFTZFPNCU4VWHK,4.412,Running Shoes For Men (Black),bonexy,Men's Footwear,379.0,4.1


* Many products have identical titles (colour variants, different sellers), so they get exactly the same score.
* *pack of 3*: the number is lost, so packs of 2 and 5 match too.
* *women floral print shirt*: unstitched shirt fabric ranks among the shirts.
* *black running shoes for men*: women's shoes match because of the sub_category "Men's Footwear".

---
# PART 2 — Evaluation

## 2.1 Evaluation metrics (point 1)

`irwa_evaluation.py` implements P@K, R@K, AP@K, F1@K, MAP, MRR and NDCG, adapted from Practice Session 2 (R@K and F1@K
are new). The main change: in the practice, every relevant document is in the evaluated list. Our AND search can miss
relevant products, so R@K, F1@K and NDCG also receive the labels of the products it did not return: missed relevant
products count in the total number of relevant products and in the ideal ranking of NDCG.

## 2.2 Validation queries (point 2)

Only the labelled products can be evaluated, so for each query we keep the labelled products in the order our engine
returns them. Labelled products that the AND search misses count as missed relevant products.

In [19]:
judgments = labels.rename(columns={"labels": "is_relevant"})[["query_id", "pid", "is_relevant"]]


def search_results_for(queries, judgments):
    """Search results in the Practice 2 format: labelled products returned by the engine, in ranking order."""
    rows = []
    for qid, query in queries.items():
        labelled = judgments[judgments["query_id"] == qid].set_index("pid")["is_relevant"]
        ranked = ix.search_tf_idf(pp.preprocess_text(query), index, tf, idf)
        for rank, (pid, score) in enumerate(ranked, start=1):
            if pid in labelled.index:
                rows.append({"query_id": qid, "pid": pid, "rank": rank,
                             "predicted_relevance": score, "is_relevant": int(labelled[pid])})
    return pd.DataFrame(rows)


search_results = search_results_for(VALIDATION_QUERIES, judgments)
for qid in VALIDATION_QUERIES:
    print(f"query {qid}: {(judgments.query_id == qid).sum()} labelled, {(search_results.query_id == qid).sum()} returned by AND")

query 1: 20 labelled, 20 returned by AND
query 2: 20 labelled, 8 returned by AND


In [20]:
validation_metrics = ev.evaluate(search_results, ks=(5, 10, 20), judgments=judgments)
with pd.option_context("display.float_format", "{:.3f}".format):
    display(validation_metrics.rename(index={"mean": "mean (MAP, MRR)"}))

metric,P,R,AP,F1,RR,NDCG,P,R,AP,F1,RR,NDCG,P,R,AP,F1,RR,NDCG
K,5,5,5,5,5,5,10,10,10,10,10,10,20,20,20,20,20,20
query_id,,,,,,,,,,,,,,,,,,
1,1.000,0.385,1.000,0.556,1.000,1.000,1.000,0.769,1.000,0.870,1.000,1.000,0.650,1.000,0.980,0.788,1.000,0.995
2,0.800,0.400,1.000,0.533,1.000,0.869,0.600,0.600,0.948,0.600,1.000,0.716,0.300,0.600,0.948,0.400,1.000,0.716
"mean (MAP, MRR)",0.900,0.392,1.000,0.544,1.000,0.934,0.800,0.685,0.974,0.735,1.000,0.858,0.475,0.800,0.964,0.594,1.000,0.855


## 2.3 Our test queries (point 3a)

We labelled the top 20 results of each query (query 5 has only 11 matches) in `test_queries_labels.csv`: 1 if the
product is the product type asked, for the gender asked, and meets every attribute of the query; 0 otherwise.
Price, rating, stock and brand are ignored. Decisions for each query:

| # | relevant only if |
|---|---|
| 3 | a formal shirt (not casual, not trousers), slim fit, with white among its colours |
| 4 | a women's shirt with a floral print (not shirt fabric, not a T-shirt) |
| 5 | women's black track pants with zip pockets (not men's, not trousers) |
| 6 | exactly a pack of 3 |
| 7 | men's running shoes with black among their colours (not lace-ups, loafers or slip-ons) |

The notebook writes the file (with empty labels) only if it does not exist, so the labels are never overwritten.

In [21]:
LABELS_FILE = Path("test_queries_labels.csv")

if not LABELS_FILE.exists():
    rows = []
    for qid, query in OUR_QUERIES.items():
        ranked = ix.search_tf_idf(pp.preprocess_text(query), index, tf, idf)
        for rank, (pid, score) in enumerate(ranked[:20], start=1):
            product = corpus.loc[pid]
            rows.append({"query_id": qid, "query": query, "rank": rank, "pid": pid, "score": round(score, 4),
                         "title": product.title, "ideal_for": product.details.get("Ideal For", ""),
                         "color": product.details.get("Color", ""), "fabric": product.details.get("Fabric", ""),
                         "sub_category": product.sub_category, "brand": product.brand_full, "url": product.url,
                         "is_relevant": None})
    pd.DataFrame(rows).to_csv(LABELS_FILE, index=False)
    print(f"{LABELS_FILE} written: fill the is_relevant column (1/0) and run the notebook again.")

our_labels = pd.read_csv(LABELS_FILE)
display(our_labels.groupby(["query_id", "query"])["is_relevant"].agg(["size", "count", "sum"])
        .rename(columns={"size": "judged", "count": "labelled", "sum": "relevant"}))

,,judged,labelled,relevant
query_id,query,,,
3,men white formal shirt slim fit,20,20,5
4,women floral print shirt,20,20,12
5,women black track pants with zip pockets,11,11,8
6,women pure cotton t-shirt pack of 3,20,20,6
7,black running shoes for men,20,20,5


Only the judged products are known, so $R@20$ would always be 1: we report $K$ = 5 and 10.

In [22]:
if our_labels["is_relevant"].isna().any():
    print("Some products have no label yet: fill test_queries_labels.csv and run the notebook again.")
else:
    our_judgments = our_labels[["query_id", "pid", "is_relevant"]].astype({"is_relevant": int})
    our_results = search_results_for(OUR_QUERIES, our_judgments)
    # the judged products must still be the engine's top results
    check = our_labels.merge(our_results[["query_id", "pid", "rank"]], on=["query_id", "pid"], how="left")
    assert (check["rank_x"] == check["rank_y"]).all(), "the ranking changed: rebuild the labels file"
    our_metrics = ev.evaluate(our_results, ks=(5, 10), judgments=our_judgments)
    with pd.option_context("display.float_format", "{:.3f}".format):
        display(our_metrics.rename(index={"mean": "mean (MAP, MRR)"}))

metric,P,R,AP,F1,RR,NDCG,P,R,AP,F1,RR,NDCG
K,5,5,5,5,5,5,10,10,10,10,10,10
query_id,,,,,,,,,,,,
3,0.600,0.600,0.917,0.600,1.000,0.699,0.300,0.600,0.917,0.400,1.000,0.699
4,0.800,0.333,1.000,0.471,1.000,0.869,0.900,0.750,0.928,0.818,1.000,0.915
5,0.800,0.500,0.950,0.615,1.000,0.854,0.700,0.875,0.909,0.778,1.000,0.891
6,0.600,0.500,0.589,0.545,0.500,0.515,0.600,1.000,0.598,0.750,0.500,0.746
7,0.200,0.200,0.500,0.200,0.500,0.214,0.300,0.600,0.341,0.400,0.500,0.414
"mean (MAP, MRR)",0.600,0.427,0.791,0.486,0.800,0.630,0.560,0.765,0.739,0.629,0.800,0.733


The comments on the metrics (point 3b) and the limitations of the system (point 3c) are in the report.